In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/term-deposit-marketing-2020.csv")

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numerical_cols = [
    "age",
    "balance",
    "day",
    #"duration",
    "campaign"
]

one_hot_encode_cols = [
    "job",
    "marital",
    "education",
    "contact",
    "month"
]

label_encode_cols = [
    "default",
    "housing",
    "loan"
]

In [18]:
from sklearn.model_selection import train_test_split

df["education"] = df["education"].replace("unknown", "missing")
X = df.drop(columns=["y", "duration"])
y = df["y"].map({"no": 0, "yes": 1})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [19]:
#feature selection of the original features (not the one-hot encoded features)

import numpy as np
from sklearn.feature_selection import mutual_info_classif
from sklearn.base import BaseEstimator, TransformerMixin


class FeatureSelector(BaseEstimator, TransformerMixin):

    def __init__(self, k="all"):
        self.k = k

    def fit(self, X, y):

        # don't modify original data
        X = X.copy()
        X_encoded = X.copy()

        feature_scores = []

        for column in X_encoded:

            # Encode categorical features using label encoding
            if column in label_encode_cols:
                X_encoded[column] = X_encoded[column].astype("category").cat.codes.to_numpy().reshape(-1, 1)
                score = mutual_info_classif(X_encoded[[column]], y, discrete_features=True, random_state=42)[0]

            # Encode categorical features using one-hot encoding
            elif column in one_hot_encode_cols:  
                encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
                X_encoded[column] = encoder.fit_transform(X_encoded[[column]])

                scores = mutual_info_classif(X_encoded[[column]], y, random_state=42)
                score = scores.max()  # Take the maximum score among the one-hot encoded features

            # Numerical feature
            else:
                score = mutual_info_classif(X[[column]],y,random_state=42)[0]

            feature_scores.append(score)

        # Calculate feature scores
        self.scores_ = np.array(feature_scores)

        # Rank features from highest to lowest score
        self.indices_ = np.argsort(self.scores_)[::-1]

        if self.k == "all":
            self.selected_indices_ = self.indices_
        else:
            self.selected_indices_ = self.indices_[:self.k]

        #change selected features to original feature names
        self.selected_features_ = X.columns[self.selected_indices_].tolist()

        return self

    #return dataset with only the selected features
    def transform(self, X):
        return X[self.selected_features_]

    def get_feature_names_out(self, input_features=None):
        return np.array(self.selected_features_)

In [20]:
from sklearn.preprocessing import OrdinalEncoder


class DynamicPreprocessor(BaseEstimator, TransformerMixin):

    def __init__(self):
        self.preprocessor_ = None

    def fit(self, X, y=None):

        # Select only the features that are present in the dataset
        selected_numerical = [col for col in numerical_cols if col in X.columns]
        selected_label = [col for col in label_encode_cols if col in X.columns]
        selected_one_hot = [col for col in one_hot_encode_cols if col in X.columns]

        self.preprocessor_ = ColumnTransformer(
            transformers=[
                ("number", StandardScaler(), selected_numerical),
                ("one_hot", OneHotEncoder(handle_unknown="ignore"), selected_one_hot),
                ("label", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), selected_label)
            ]
        )

        self.preprocessor_.fit(X, y)

        return self

    # return the transformed dataset with only the selected features
    def transform(self, X):
        return self.preprocessor_.transform(X)

    def get_feature_names_out(self, input_features=None):
        return self.preprocessor_.get_feature_names_out(input_features)

"Random Forest": {
        "pipeline": Pipeline([
            ("preprocessor", preprocessor),
            ("model", RandomForestClassifier(random_state=42, class_weight="balanced"))
        ]),
        "params": {
            "model__n_estimators": [50, 100, 150, 200],
            "model__max_depth": [1, 2, 5, 10],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4]
        }
    },

In [ ]:
#setup baseline models

from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from xgboost import XGBClassifier


#variables for models

#feature selector
feature_selector = FeatureSelector()

#preprocessor
preprocessor = DynamicPreprocessor()

#scale pos weight for xgboost
scale_pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]


models = {
    "Logistic Regression": {
        "pipeline": Pipeline([
            ("select", feature_selector),
            ("preprocessor", preprocessor),
            ("model", LogisticRegression(random_state=42, max_iter=1000, class_weight="balanced"))
        ]),
        "params": {
            "model__C": [0.01, 0.1, 1, 10],
            "select__k": [3, 5, 7, 10, "all"]
        }
    }, 

    "XGBoost": {
            "pipeline": Pipeline([
                ("select", feature_selector),
                ("preprocessor", preprocessor),
                ("model", XGBClassifier(random_state=42, scale_pos_weight=scale_pos_weight))
            ]),
            "params": {
                "model__max_depth": [1, 2, 3, 4, 6, 10],
                "model__learning_rate": [0.05, 0.1, 0.2],
                "model__n_estimators": [50, 100, 200],
                "select__k": [3, 5, 7, 10, "all"]
            }
        },

    

        "KNN": {
        "pipeline": Pipeline([
            ("select", feature_selector),
            ("preprocessor", preprocessor),
            ("model", KNeighborsClassifier())
        ]),
        "params": {
            "model__n_neighbors": [3, 5, 7, 9, 11],
            "model__weights": ["uniform", "distance"],
            "select__k": [3, 5, 7, 10, "all"]
        }
    },


    "SVM": {
        "pipeline": Pipeline([
            ("select", feature_selector),
            ("preprocessor", preprocessor),
            ("model", SVC(random_state=42, class_weight="balanced", probability=True, kernel="linear"))
        ]),
        "params": {
            "model__C": [0.01, 0.1, 1, 10],
            "select__k": [3, 5, 7, 10, "all"]
        }
    }
}
    


In [22]:
from sklearn.model_selection import StratifiedKFold


cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [27]:
import time
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, make_scorer, precision_score, recall_score
from sklearn.model_selection import GridSearchCV, cross_val_score, cross_validate


results = []

# Metrics to calculate during cross-validation
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

grid_search_results = {}


for name, config in models.items():
    print(f"Training model: {name}")

    # Start timer
    start_time = time.perf_counter()

    # Grid search
    grid_search = GridSearchCV(config["pipeline"], config["params"], cv=cv, scoring=scoring, refit="f1", n_jobs=-1)

    grid_search.fit(X_train, y_train)

    # Store the complete GridSearchCV object
    grid_search_results[name] = grid_search

    best_model = grid_search.best_estimator_

    best_index = grid_search.best_index_

    # Get feature selection information
    selector = best_model.named_steps["select"]

    selected_features = selector.selected_features_
    feature_scores = selector.scores_

    # Evaluate best model
    #cv_results = cross_validate(best_model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    
    elapsed_time = time.perf_counter() - start_time

    # Record results
    results.append({
        "Model": name,
        "Accuracy": grid_search.cv_results_["mean_test_accuracy"][best_index],
        "Precision": grid_search.cv_results_["mean_test_precision"][best_index],
        "Recall": grid_search.cv_results_["mean_test_recall"][best_index],
        "F1 Score": grid_search.cv_results_["mean_test_f1"][best_index],
        "Training Time (seconds)": elapsed_time,
        "Selected Features": selected_features,
        "Best Parameters": grid_search.best_params_
    })
    
    
    
    
    """results.append({
        "Model": name,
        "Accuracy": cv_results["test_accuracy"].mean(),
        "Precision": cv_results["test_precision"].mean(),
        "Recall": cv_results["test_recall"].mean(),
        "F1 Score": cv_results["test_f1"].mean(),
        "Training Time (seconds)": elapsed_time,
        "Selected Features": selected_features,
        "Feature Scores": feature_scores,
        "Best Parameters": grid_search.best_params_
    })"""


Training model: Logistic Regression
Training model: XGBoost
Training model: KNN


In [28]:
results_df = pd.DataFrame(results)

print(results_df.to_string(index=False))

              Model  Accuracy  Precision   Recall  F1 Score  Training Time (seconds)                                                                              Selected Features                                                                                        Best Parameters
Logistic Regression  0.699469   0.126846 0.535179  0.205052                 8.234184                                                            [contact, month, age, balance, day]                                                                       {'model__C': 10, 'select__k': 5}
            XGBoost  0.846656   0.190903 0.345686  0.245922               118.273762 [contact, month, age, balance, day, campaign, marital, education, job, housing, loan, default] {'model__learning_rate': 0.05, 'model__max_depth': 10, 'model__n_estimators': 200, 'select__k': 'all'}
                KNN  0.926156   0.460612 0.107471  0.174005                55.881043                                                            [contac

In [29]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

results = []

for name, grid_search in grid_search_results.items():

    # Best pipeline found during GridSearchCV
    final_model = grid_search.best_estimator_

    # Predict ONLY on the untouched test set
    y_pred = final_model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)

    results.append({
        "Model": name,
        "CV F1": grid_search.best_score_,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Confusion Matrix": cm,
        "Best Parameters": grid_search.best_params_
    })

results_df = pd.DataFrame(results)
print(results_df)

                 Model     CV F1  Accuracy  Precision    Recall  F1 Score  \
0  Logistic Regression  0.205052   0.71700   0.139803  0.564767  0.224126   
1              XGBoost  0.245922   0.84550   0.204320  0.392055  0.268639   
2                  KNN  0.174005   0.92625   0.449541  0.084629  0.142442   

             Confusion Matrix  \
0  [[5409, 2012], [252, 327]]   
1   [[6537, 884], [352, 227]]   
2     [[7361, 60], [530, 49]]   

                                     Best Parameters  
0                   {'model__C': 10, 'select__k': 5}  
1  {'model__learning_rate': 0.05, 'model__max_dep...  
2  {'model__n_neighbors': 5, 'model__weights': 'u...  


try dropping duration to see the precision, recall, and f1 results

try:
 - svm
 - knn
 - random forest
 - xgboost

In [ ]:
"""

Model               Accuracy  Precision   Recall  F1 Score  Training Time (seconds)                                                                              Selected Features                                                                                                                                                                                                                                              Feature Scores                                                         Best Parameters
Logistic Regression  0.657656   0.121934 0.601200  0.202746                19.387307 [month, housing, contact, age, day, balance, marital, education, job, campaign, default, loan] [0.006338099325582158, 0.0025700594961286516, 0.0035518988493001835, 0.002606710993709349, 2.2253229923885343e-05, 0.004185981395054217, 0.008903672379657612, 0.0, 0.00683381606086142, 0.006323285113879251, 0.012808924633678664, 0.0015177271233184353]                                   {'model__C': 0.1, 'select__k': 'all'}
            XGBoost  0.846094   0.188685 0.341812  0.243071                48.372505                [month, housing, contact, age, day, balance, marital, education, job, campaign] [0.006338099325582158, 0.0025700594961286516, 0.0035518988493001835, 0.002606710993709349, 2.2253229923885343e-05, 0.004185981395054217, 0.008903672379657612, 0.0, 0.00683381606086142, 0.006323285113879251, 0.012808924633678664, 0.0015177271233184353]  {'model__learning_rate': 0.1, 'model__max_depth': 10, 'select__k': 10}
                KNN  0.901375   0.214903 0.129472  0.159716               120.057894                                                            [month, housing, contact, age, day] [0.006338099325582158, 0.0025700594961286516, 0.0035518988493001835, 0.002606710993709349, 2.2253229923885343e-05, 0.004185981395054217, 0.008903672379657612, 0.0, 0.00683381606086142, 0.006323285113879251, 0.012808924633678664, 0.0015177271233184353] {'model__n_neighbors': 3, 'model__weights': 'distance', 'select__k': 5}
                SVM  0.736313   0.143635 0.532141  0.226126              3045.730321 [month, housing, contact, age, day, balance, marital, education, job, campaign, default, loan] [0.006338099325582158, 0.0025700594961286516, 0.0035518988493001835, 0.002606710993709349, 2.2253229923885343e-05, 0.004185981395054217, 0.008903672379657612, 0.0, 0.00683381606086142, 0.006323285113879251, 0.012808924633678664, 0.0015177271233184353]                                     {'model__C': 1, 'select__k': 'all'}



Note:

SVM consistently takes a much longer time to train than the other models, and does not show significant improvement.
It is for this reason that I have commented it out of the models dictionary. 



Notes:
treat education unknown as missing - "n/a"
try different kernel for svm

"""